# 27 / 29 — Bagging (3 graines) du champion et du pseudo

Hypothèse : stabiliser par bagging avant le blend. Verdict : **négatif** — la graine 42 est déjà
un bon tirage, la moyenner avec d'autres régresse vers la moyenne.
- 27_bagged_w75/w80 (raw) : non soumis.
- **29_bagrank_w70** : **LB 0.354940** (vs 0.357804 single-seed) — piste refermée.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
from catboost import CatBoostClassifier
from src import config as C
from src.utils import op03_mask, seed_everything, make_submission
from src.features.temporal import balance_features, recency_features
from src.features.behavioral import behavioral_features
from src.encoding import oof_target_encode_train, fit_target_map, apply_target_map, recent_target_rate
seed_everything(42)
DATA = ROOT / "data"
train = pd.read_csv(DATA / "train.csv"); test = pd.read_csv(DATA / "test.csv")
op03 = op03_mask(train).to_numpy(); y_all = train[C.TARGET].to_numpy()
te_op = op03_mask(test).to_numpy()
EPS = 1e-6; W = (5, 10, 20); SM = 30
def rowf(df):
    f = pd.DataFrame(index=df.index)
    f["amount_log1p"] = np.log1p(np.maximum(df[C.AMOUNT], 0))
    f["amount_vs_origin_before"] = df[C.AMOUNT] / (np.abs(df[C.ORIGIN_BAL_BEFORE]) + EPS)
    f["amount_vs_dest_before"] = df[C.AMOUNT] / (np.abs(df[C.DEST_BAL_BEFORE]) + EPS)
    f["origin_balance_before"] = df[C.ORIGIN_BAL_BEFORE]; f["dest_balance_before"] = df[C.DEST_BAL_BEFORE]
    return pd.concat([f, balance_features(df)], axis=1)
def bb(df, ref):
    X = rowf(df).reset_index(drop=True)
    for c in [C.ORIGIN_ACCT, C.DEST_ACCT]:
        X[f"fq_{c}"] = df[c].map(ref[c].value_counts(normalize=True)).fillna(0).values
    return pd.concat([X, behavioral_features(df, ref).reset_index(drop=True),
                      recency_features(df, ref).reset_index(drop=True),
                      recent_target_rate(df, ref, C.ORIGIN_ACCT, C.PERIOD, C.TARGET, W).reset_index(drop=True)], axis=1)
def ftr(df, ref):
    X = bb(df, ref); X["te"] = oof_target_encode_train(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=SM); return X
def fap(df, ref):
    X = bb(df, ref); mp, gm = fit_target_map(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=SM)
    X["te"] = apply_target_map(df, C.ORIGIN_ACCT, mp, gm); return X
def cat(seed=42):
    return CatBoostClassifier(loss_function="Logloss", eval_metric="PRAUC", depth=6,
                              learning_rate=0.05, iterations=600, random_seed=seed, verbose=False)
def rk(x):
    return np.argsort(np.argsort(x)) / (len(x) - 1)
def out(name, p_op):
    full = np.zeros(len(test)); full[te_op] = p_op
    print("écrit :", make_submission(test[C.ID], full, name))
ref0 = train.iloc[np.where(op03)[0]]; y0 = y_all[op03]
test_op = test.iloc[np.where(te_op)[0]].copy()
print("setup OK |", op03.sum(), "op03 train /", te_op.sum(), "op03 test")

In [ ]:
def train_champion(seed=42):
    """Champion CatBoost (config nb08) entraîné sur tout le train op_03."""
    return cat(seed).fit(ftr(ref0, ref0), y0).predict_proba(fap(test_op, ref0))[:, 1]

def pseudo_from(pch, thr_f=0.98, thr_l=0.02, seed=42):
    """Pseudo-labeling 1 cycle (cf. nb23) puis réentraînement."""
    mfm = pch > thr_f; mlm = pch < thr_l
    pse = test_op.iloc[np.where(mfm | mlm)[0]].copy()
    pse[C.TARGET] = (pch[mfm | mlm] > thr_f).astype(float)
    aug = pd.concat([ref0, pse], ignore_index=True)
    print(f"pseudo ({thr_f}/{thr_l}, seed {seed}) : {int(mfm.sum())} fraudes / {int(mlm.sum())} légitimes")
    m = cat(seed).fit(ftr(aug, aug), aug[C.TARGET].to_numpy())
    return m.predict_proba(fap(test_op, aug))[:, 1]

In [ ]:
SEEDS = [42, 1, 7]
Xtr0 = ftr(ref0, ref0); Xte0 = fap(test_op, ref0)
pch_bag = np.mean([cat(s).fit(Xtr0, y0).predict_proba(Xte0)[:, 1] for s in SEEDS], axis=0)
mfm = pch_bag > 0.98; mlm = pch_bag < 0.02
pse = test_op.iloc[np.where(mfm | mlm)[0]].copy()
pse[C.TARGET] = (pch_bag[mfm | mlm] > 0.98).astype(float)
aug = pd.concat([ref0, pse], ignore_index=True)
print(f"pseudo baggé : {int(mfm.sum())} fraudes / {int(mlm.sum())} légitimes")
Xtra = ftr(aug, aug); Xtea = fap(test_op, aug); ya = aug[C.TARGET].to_numpy()
p1_bag = np.mean([cat(s).fit(Xtra, ya).predict_proba(Xtea)[:, 1] for s in SEEDS], axis=0)

## 27 — raw / 29 — rank

In [ ]:
out("27_bagged_w75", 0.25 * pch_bag + 0.75 * p1_bag)
out("27_bagged_w80", 0.20 * pch_bag + 0.80 * p1_bag)
for wp in [65, 68, 70]:
    out(f"29_bagrank_w{wp}", (1 - wp / 100) * rk(pch_bag) + (wp / 100) * rk(p1_bag))